# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 255.05it/s]


2026-04-07 19:42:35.879 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:907 - Data batch-empirical estimation of propensity score.


2026-04-07 19:42:35.887 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:956 - Data prediction of expected reward based on gbm model.


2026-04-07 19:42:37.350 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1073 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-04-07 19:43:07.954 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-04-07 19:43:08.264 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-04-07 19:43:08.573 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.515358,0.482281,0.548567,0.016979,b-ipw,reward_0
1,0.495371,0.494465,0.496269,0.000459,dm,reward_0
2,0.510985,0.479517,0.543191,0.016249,dr,reward_0
3,0.495371,0.494434,0.496294,0.000467,dros-opt,reward_0
4,0.510985,0.479574,0.543401,0.016197,dros-pess,reward_0
5,0.510879,0.477946,0.544353,0.016783,ipw,reward_0
6,0.510792,0.477718,0.544462,0.016881,rep,reward_0
7,0.510983,0.479398,0.543530,0.016392,sndr,reward_0
8,0.510807,0.478256,0.543847,0.016660,snips,reward_0
9,0.510985,0.478826,0.542613,0.016145,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 288.86it/s]


2026-04-07 19:43:09.044 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1305 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<08:17,  2.01it/s]

...

2026-04-07 19:43:56.429 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-04-07 19:43:56.741 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-04-07 19:43:57.049 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1191 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.486484,0.436027,0.542215,0.026761,b-ipw,reward_0
1,0.495743,0.494820,0.496619,0.000459,dm,reward_0
2,0.509502,0.465190,0.553887,0.022416,dr,reward_0
3,0.495743,0.494834,0.496652,0.000462,dros-opt,reward_0
4,0.509502,0.466088,0.554554,0.022383,dros-pess,reward_0
5,0.510246,0.456967,0.565574,0.027843,ipw,reward_0
6,0.510246,0.456967,0.566504,0.028171,rep,reward_0
7,0.509502,0.464690,0.553131,0.022734,sndr,reward_0
8,0.510246,0.456967,0.567623,0.028105,snips,reward_0
9,0.509502,0.464541,0.554223,0.022696,sg-dr,reward_0
